# GR00T N1.6 — Step 1: Data Preparation

This notebook downloads BridgeData V2 from HuggingFace, generates 7-DoF actions via optical flow,
saves intermediate episode folders (like the OpenVLA pipeline), converts to LeRobot V2 format,
validates the dataset, and uploads to S3 for the SageMaker training job.

**Pipeline:**
1. Download BridgeData V2 → save episode folders (images, actions.npy, language.txt)
2. Validate intermediate dataset (check for NaN, shape, missing files)
3. Convert episode folders → LeRobot V2 format (parquet + MP4 + metadata)
4. Upload to S3

**Runtime:** ~30-45 min for 600 episodes

## 1. Environment Setup

In [1]:
# Install dependencies (run once)
!pip install -q datasets pyarrow opencv-python-headless Pillow numpy tqdm sagemaker boto3

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.5.0 requires nvidia-ml-py3<8.0,>=7.352.0, which is not installed.
autogluon-timeseries 1.5.0 requires chronos-forecasting<2.4,>=2.2.2, which is not installed.
autogluon-timeseries 1.5.0 requires einops<1,>=0.7, which is not installed.
aiobotocore 2.22.0 requires botocore<1.37.4,>=1.37.2, but you have botocore 1.42.91 which is incompatible.
amazon-sagemaker-jupyter-ai-q-developer 1.2.9 requires numpy<=2.0.1, but you have numpy 2.4.4 which is incompatible.
amazon-sagemaker-sql-magic 0.1.4 requires numpy<2, but you have numpy 2.4.4 which is incompatible.
autogluon-common 1.5.0 requires numpy<2.4.0,>=1.25.0, but you have numpy 2.4.4 which is incompatible.
autogluon-core 1.5.0 requires numpy<2.4.0,>=1.25.0, but you have numpy 2.4.4 which is incompatible.
autogluon-features 1.5.0 requires numpy<2.

### Authentication Setup

Configure Hugging Face authentication to access the BridgeData V2 dataset.

In [2]:
from getpass import getpass
from huggingface_hub import login

# Prompt for Hugging Face token (input is hidden)
hf_token = getpass("Enter your Hugging Face token: ")
login(token=hf_token)

Enter your Hugging Face token:  ········


In [3]:
import os
import json
import subprocess
import tempfile
from pathlib import Path

import boto3
import cv2
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq
import sagemaker
from datasets import load_dataset
from PIL import Image
from tqdm import tqdm

## 2. Configuration

### SageMaker Session

Initialize SageMaker session for S3 access.

**IAM Permissions:** The SageMaker execution role should follow the principle of least privilege. At minimum, it needs:
- `s3:GetObject` and `s3:PutObject` scoped to the specific training data and output bucket paths
- `sagemaker:CreateTrainingJob` for launching training jobs

In [4]:
from sagemaker.core.helper import session_helper

sagemaker_session = session_helper.Session()
region = sagemaker_session.boto_region_name
account_id = boto3.client("sts").get_caller_identity()["Account"]
bucket_name = sagemaker_session.default_bucket()
default_prefix = sagemaker_session.default_bucket_prefix

print(f"Region: {region}")
print(f"Account: {account_id}")
print(f"Bucket: {bucket_name}")
print(f"Default prefix: {default_prefix}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
Region: us-east-1
Account: 783764584149
Bucket: sagemaker-us-east-1-783764584149
Default prefix: None


In [5]:
# Dataset parameters
MAX_EPISODES = 600          # Number of training episodes (use 10 for quick test)
ACTION_DIM = 7              # 7-DoF: x, y, z, roll, pitch, yaw, gripper

# Local paths
RAW_DATASET_DIR = "./bridge_synthetic_dataset"   # Intermediate episode folders
LEROBOT_DATASET_DIR = "./datasets/bridge_lerobot" # Final LeRobot V2 format

# S3 configuration
if default_prefix:
    S3_PREFIX = f"{default_prefix}/groot-finetuning/datasets/bridge_lerobot"
else:
    S3_PREFIX = "groot-finetuning/datasets/bridge_lerobot"

print(f"Episodes: {MAX_EPISODES}")
print(f"Raw dataset: {RAW_DATASET_DIR}")
print(f"LeRobot dataset: {LEROBOT_DATASET_DIR}")
print(f"S3 destination: s3://{bucket_name}/{S3_PREFIX}")

Episodes: 600
Raw dataset: ./bridge_synthetic_dataset
LeRobot dataset: ./datasets/bridge_lerobot
S3 destination: s3://sagemaker-us-east-1-783764584149/groot-finetuning/datasets/bridge_lerobot


## 3. Download BridgeData & Generate Actions (Optical Flow)

This step downloads `VyoJ/BridgeData-V2-Scripted-Images` from HuggingFace and saves each episode
as a folder with images, actions (via optical flow), and language instruction.

This follows the same pattern as the OpenVLA pipeline (`fix_synthetic_dataset_fast.py`).

In [6]:
def estimate_actions_from_optical_flow(frames, action_dim=7):
    """
    Estimate 7-DoF actions from optical flow between consecutive frames.
    Returns actions of shape (T, action_dim).
    """
    actions = []
    for i in range(len(frames) - 1):
        img1 = np.array(frames[i].convert("RGB"))
        img2 = np.array(frames[i + 1].convert("RGB"))
        gray1 = cv2.cvtColor(img1, cv2.COLOR_RGB2GRAY)
        gray2 = cv2.cvtColor(img2, cv2.COLOR_RGB2GRAY)
        flow = cv2.calcOpticalFlowFarneback(
            gray1, gray2, None,
            pyr_scale=0.5, levels=3, winsize=15,
            iterations=3, poly_n=5, poly_sigma=1.2, flags=0,
        )
        mean_flow = np.mean(flow, axis=(0, 1))
        action = np.zeros(action_dim, dtype=np.float32)
        action[0] = mean_flow[0] / 100.0  # dx
        action[1] = mean_flow[1] / 100.0  # dy
        # dz, roll, pitch, yaw, gripper = 0 (can't infer from 2D)
        actions.append(action)
    # Final "stop" action
    actions.append(np.zeros(action_dim, dtype=np.float32))
    return np.array(actions, dtype=np.float32)

In [7]:
print("=== Downloading BridgeData V2 Scripted Images ===")
ds = load_dataset("VyoJ/BridgeData-V2-Scripted-Images")
data = ds["train"]
num_episodes = min(len(data), MAX_EPISODES)
print(f"Source: {len(data)} episodes, using {num_episodes}")

os.makedirs(RAW_DATASET_DIR, exist_ok=True)

print(f"\n=== Generating episode folders ===")
for idx in tqdm(range(num_episodes), desc="Generating"):
    sample = data[idx]
    episode_dir = os.path.join(RAW_DATASET_DIR, f"episode_{idx:06d}")
    os.makedirs(os.path.join(episode_dir, "images"), exist_ok=True)

    # Save images
    frames = [sample["first_image"], sample["intermediate_image"], sample["frame_43_image"]]
    frames = [f for f in frames if f is not None]
    for t, img in enumerate(frames):
        img.save(os.path.join(episode_dir, "images", f"{t:06d}.jpg"))

    # Generate actions via optical flow
    instruction = "move object to target"
    actions = estimate_actions_from_optical_flow(frames, ACTION_DIM)

    # Validate before saving
    if actions is None or actions.size == 0:
        actions = np.zeros((len(frames), ACTION_DIM), dtype=np.float32)
    if np.any(np.isnan(actions)) or np.any(np.isinf(actions)):
        actions = np.nan_to_num(actions, nan=0.0, posinf=0.0, neginf=0.0)

    # Save actions, language, metadata
    np.save(os.path.join(episode_dir, "actions.npy"), actions)
    with open(os.path.join(episode_dir, "language.txt"), "w") as f:
        f.write(instruction)
    with open(os.path.join(episode_dir, "metadata.json"), "w") as f:
        json.dump({"length": len(frames), "method": "optical_flow"}, f)

print(f"\nRaw dataset saved to: {RAW_DATASET_DIR}")
print(f"Episodes: {num_episodes}")

=== Downloading BridgeData V2 Scripted Images ===


Resolving data files:   0%|          | 0/26407 [00:00<?, ?it/s]

Source: 8802 episodes, using 600

=== Generating episode folders ===


Generating: 100%|██████████| 600/600 [01:32<00:00,  6.49it/s]


Raw dataset saved to: ./bridge_synthetic_dataset
Episodes: 600


## 4. Validate Intermediate Dataset

Check all episodes for valid actions (no NaN, correct shape, no missing files).
Same validation pattern as `check_synthetic_dataset.py` from the OpenVLA pipeline.

In [8]:
episodes = sorted([d for d in os.listdir(RAW_DATASET_DIR)
                   if os.path.isdir(os.path.join(RAW_DATASET_DIR, d))
                   and d.startswith("episode_")])

valid_count = 0
invalid_count = 0
missing_count = 0

for ep_name in episodes:
    ep_path = os.path.join(RAW_DATASET_DIR, ep_name)
    actions_file = os.path.join(ep_path, "actions.npy")

    if not os.path.exists(actions_file):
        missing_count += 1
        continue

    try:
        actions = np.load(actions_file, allow_pickle=True)
        if actions.dtype == object:
            actions = np.array(actions.tolist(), dtype=np.float32)

        has_images = len([f for f in os.listdir(os.path.join(ep_path, "images"))
                         if f.endswith((".jpg", ".png"))]) > 0
        has_lang = os.path.exists(os.path.join(ep_path, "language.txt"))
        no_nan = not np.any(np.isnan(actions)) and not np.any(np.isinf(actions))

        if has_images and has_lang and no_nan and actions.shape[1] == 7:
            valid_count += 1
        else:
            invalid_count += 1
    except:
        invalid_count += 1

print(f"Total episodes: {len(episodes)}")
print(f"✅ Valid: {valid_count} ({100*valid_count/len(episodes):.1f}%)")
print(f"⚠️  Invalid: {invalid_count}")
print(f"❌ Missing: {missing_count}")

if valid_count == len(episodes):
    print("\n✅ All episodes have valid actions! Ready to convert.")
else:
    print(f"\n⚠️  {invalid_count + missing_count} episodes have issues.")

Total episodes: 600
✅ Valid: 600 (100.0%)
⚠️  Invalid: 0
❌ Missing: 0

✅ All episodes have valid actions! Ready to convert.


In [9]:
# Show a sample episode
sample_ep = os.path.join(RAW_DATASET_DIR, episodes[0])
actions = np.load(os.path.join(sample_ep, "actions.npy"))
with open(os.path.join(sample_ep, "language.txt")) as f:
    lang = f.read().strip()
images = sorted([f for f in os.listdir(os.path.join(sample_ep, "images")) if f.endswith(".jpg")])

print(f"Sample episode: {episodes[0]}")
print(f"  Images: {len(images)} files")
print(f"  Actions shape: {actions.shape}")
print(f"  Actions:\n{actions}")
print(f"  Min: {np.min(actions):.6f}, Max: {np.max(actions):.6f}")
print(f"  Language: '{lang}'")

Sample episode: episode_000000
  Images: 3 files
  Actions shape: (3, 7)
  Actions:
[[-0.00182916 -0.00047819  0.          0.          0.          0.
   0.        ]
 [-0.00106693 -0.01065828  0.          0.          0.          0.
   0.        ]
 [ 0.          0.          0.          0.          0.          0.
   0.        ]]
  Min: -0.010658, Max: 0.000000
  Language: 'move object to target'


## 5. Convert to LeRobot V2 Format

Convert the validated episode folders to LeRobot V2 format (parquet + MP4 + metadata).
This is the format GR00T N1.6 expects for fine-tuning.

In [10]:
TARGET_FRAMES = 30  # Frames per episode after interpolation
FPS = 5

def interpolate_frames(frames, target_count=30):
    """Interpolate between frames to create a longer episode via linear blending."""
    if len(frames) >= target_count:
        return frames[:target_count]
    result = []
    n_orig = len(frames)
    for i in range(target_count):
        src_pos = i * (n_orig - 1) / (target_count - 1)
        src_idx = int(src_pos)
        frac = src_pos - src_idx
        if src_idx >= n_orig - 1:
            result.append(frames[-1])
        elif frac < 0.01:
            result.append(frames[src_idx])
        else:
            img1 = np.array(frames[src_idx].convert("RGB"), dtype=np.float32)
            img2 = np.array(frames[src_idx + 1].convert("RGB"), dtype=np.float32)
            blended = ((1 - frac) * img1 + frac * img2).astype(np.uint8)
            result.append(Image.fromarray(blended))
    return result

def images_to_video(images, output_path, fps=5):
    """Convert a list of PIL images to an MP4 video using ffmpeg."""
    with tempfile.TemporaryDirectory() as tmpdir:
        for i, img in enumerate(images):
            if not isinstance(img, Image.Image):
                img = Image.fromarray(np.array(img))
            img.save(os.path.join(tmpdir, f"frame_{i:06d}.png"))
        cmd = [
            "ffmpeg", "-y", "-framerate", str(fps),
            "-i", os.path.join(tmpdir, "frame_%06d.png"),
            "-c:v", "libx264", "-pix_fmt", "yuv420p",
            "-crf", "23", "-preset", "fast", output_path,
        ]
        subprocess.run(cmd, capture_output=True, check=True)

In [11]:
# Create LeRobot V2 output dirs
output_path = Path(LEROBOT_DATASET_DIR)
data_dir = output_path / "data" / "chunk-000"
video_dir = output_path / "videos" / "chunk-000" / "observation.images.front"
meta_dir = output_path / "meta"
for d in [data_dir, video_dir, meta_dir]:
    d.mkdir(parents=True, exist_ok=True)

episodes_meta = []
tasks_set = {}
total_frames = 0
skipped = 0
all_actions, all_states, all_timestamps = [], [], []

print("=== Converting episode folders to LeRobot V2 ===")
for ep_idx, ep_name in enumerate(tqdm(episodes, desc="Converting")):
    ep_path = os.path.join(RAW_DATASET_DIR, ep_name)

    # Load images from episode folder
    image_dir = os.path.join(ep_path, "images")
    image_files = sorted([f for f in os.listdir(image_dir) if f.endswith((".jpg", ".png"))])
    frames = [Image.open(os.path.join(image_dir, f)) for f in image_files]

    if len(frames) < 2:
        skipped += 1
        continue

    # Interpolate to 30 frames (GR00T needs >= 16 for action horizon)
    frames = interpolate_frames(frames, target_count=TARGET_FRAMES)
    num_frames = len(frames)

    # Re-generate actions for interpolated frames (optical flow on 30 frames, not 3)
    actions = estimate_actions_from_optical_flow(frames, ACTION_DIM)
    states = actions.copy()  # pseudo-state (no proprioceptive state in this dataset)

    # Load language
    with open(os.path.join(ep_path, "language.txt")) as f:
        instruction = f.read().strip()
    if instruction not in tasks_set:
        tasks_set[instruction] = len(tasks_set)
    task_index = tasks_set[instruction]

    # Collect for stats
    all_actions.extend(actions.tolist())
    all_states.extend(states.tolist())
    all_timestamps.extend([fi / FPS for fi in range(num_frames)])

    # Write parquet
    rows = []
    for fi in range(num_frames):
        rows.append({
            "observation.state": states[fi].tolist(),
            "action": actions[fi].tolist(),
            "episode_index": ep_idx, "frame_index": fi,
            "index": total_frames + fi, "task_index": task_index,
            "timestamp": fi / FPS,
        })
    pq.write_table(pa.Table.from_pylist(rows), data_dir / f"episode_{ep_idx:06d}.parquet")

    # Write video
    try:
        images_to_video(frames, str(video_dir / f"episode_{ep_idx:06d}.mp4"), fps=FPS)
    except Exception as e:
        print(f"Warning: Video failed for episode {ep_idx}: {e}")
        skipped += 1
        continue

    episodes_meta.append({"episode_index": ep_idx, "tasks": [instruction], "length": num_frames})
    total_frames += num_frames

print(f"\nConverted {len(episodes_meta)} episodes ({skipped} skipped), {total_frames} total frames")

=== Converting episode folders to LeRobot V2 ===


Converting: 100%|██████████| 600/600 [53:35<00:00,  5.36s/it]


Converted 600 episodes (0 skipped), 18000 total frames


## 6. Write LeRobot V2 Metadata

In [12]:
# info.json
info = {
    "codebase_version": "v2.1", "robot_type": "bridge_v2",
    "total_episodes": len(episodes_meta), "total_frames": total_frames,
    "total_tasks": len(tasks_set), "chunks_size": 1000, "fps": FPS,
    "splits": {"train": f"0:{len(episodes_meta)}"},
    "data_path": "data/chunk-{episode_chunk:03d}/episode_{episode_index:06d}.parquet",
    "video_path": "videos/chunk-{episode_chunk:03d}/{video_key}/episode_{episode_index:06d}.mp4",
    "features": {
        "action": {"dtype": "float32", "shape": [7], "names": ["x","y","z","roll","pitch","yaw","gripper"]},
        "observation.state": {"dtype": "float32", "shape": [7], "names": ["x","y","z","roll","pitch","yaw","gripper"]},
        "observation.images.front": {
            "dtype": "video", "shape": [480,640,3], "names": ["height","width","channels"],
            "info": {"video.height":480,"video.width":640,"video.codec":"av1","video.pix_fmt":"yuv420p",
                     "video.is_depth_map":False,"video.fps":FPS,"video.channels":3,"has_audio":False},
        },
        "timestamp": {"dtype":"float32","shape":[1],"names":None},
        "frame_index": {"dtype":"int64","shape":[1],"names":None},
        "episode_index": {"dtype":"int64","shape":[1],"names":None},
        "index": {"dtype":"int64","shape":[1],"names":None},
        "task_index": {"dtype":"int64","shape":[1],"names":None},
    },
    "total_chunks": 1, "total_videos": len(episodes_meta),
}
with open(meta_dir / "info.json", "w") as f:
    json.dump(info, f, indent=2)

# episodes.jsonl
with open(meta_dir / "episodes.jsonl", "w") as f:
    for ep in episodes_meta:
        f.write(json.dumps(ep) + "\n")

# tasks.jsonl
with open(meta_dir / "tasks.jsonl", "w") as f:
    for task_text, task_idx in sorted(tasks_set.items(), key=lambda x: x[1]):
        f.write(json.dumps({"task_index": task_idx, "task": task_text}) + "\n")

# modality.json
modality = {
    "video": {"front": {"original_key": "observation.images.front"}},
    "state": {"arm": {"start": 0, "end": 7}},
    "action": {"arm": {"start": 0, "end": 7}},
    "annotation": {"human.task_description": {"original_key": "task_index"}},
}
with open(meta_dir / "modality.json", "w") as f:
    json.dump(modality, f, indent=2)

# stats.json
all_actions_np = np.array(all_actions, dtype=np.float32)
all_states_np = np.array(all_states, dtype=np.float32)
all_timestamps_np = np.array(all_timestamps, dtype=np.float32).reshape(-1, 1)

def compute_stats(arr):
    return {
        "mean": np.mean(arr, axis=0).tolist(), "std": np.std(arr, axis=0).tolist(),
        "min": np.min(arr, axis=0).tolist(), "max": np.max(arr, axis=0).tolist(),
        "q01": np.percentile(arr, 1, axis=0).tolist(), "q99": np.percentile(arr, 99, axis=0).tolist(),
    }

stats = {
    "action": compute_stats(all_actions_np),
    "observation.state": compute_stats(all_states_np),
    "timestamp": compute_stats(all_timestamps_np),
}
with open(meta_dir / "stats.json", "w") as f:
    json.dump(stats, f, indent=2)

print(f"LeRobot V2 dataset saved to {LEROBOT_DATASET_DIR}")
print(f"  Episodes: {len(episodes_meta)}")
print(f"  Total frames: {total_frames}")

LeRobot V2 dataset saved to ./datasets/bridge_lerobot
  Episodes: 600
  Total frames: 18000


## 7. Upload Dataset to S3

In [13]:
s3_client = boto3.client("s3")
local_root = Path(LEROBOT_DATASET_DIR)

files = [f for f in local_root.rglob("*") if f.is_file()]
print(f"Uploading {len(files)} files to s3://{bucket_name}/{S3_PREFIX}/")

# Note: The SageMaker default bucket has server-side encryption (SSE-S3) enabled by default.
# For custom buckets, enable server-side encryption with ExtraArgs={'ServerSideEncryption': 'aws:kms'}
for f in tqdm(files, desc="Uploading to S3"):
    relative = f.relative_to(local_root)
    s3_key = f"{S3_PREFIX}/{relative}"
    s3_client.upload_file(str(f), bucket_name, s3_key)

print(f"\nDataset uploaded to: s3://{bucket_name}/{S3_PREFIX}/")
print("You can now run the training notebook (02_training_job.ipynb)")

Uploading 1205 files to s3://sagemaker-us-east-1-783764584149/groot-finetuning/datasets/bridge_lerobot/


Uploading to S3: 100%|██████████| 1205/1205 [00:56<00:00, 21.41it/s]


Dataset uploaded to: s3://sagemaker-us-east-1-783764584149/groot-finetuning/datasets/bridge_lerobot/
You can now run the training notebook (02_training_job.ipynb)
